# Fine-tuning EduNusa (Gemma 2B + LoRA)

Notebook ini untuk fine-tuning model dasar `gemma-2-2b` menjadi **EduNusa**, asisten belajar AI untuk platform Belajar 3T, menggunakan [Unsloth](https://github.com/unslothai/unsloth) (LoRA, hemat memori) di Google Colab.

**Langkah besar di notebook ini:**
1. Install Unsloth dan dependency
2. Upload & load `dataset-final.jsonl` (hasil dari `prepare-dataset.js`)
3. Konfigurasi LoRA
4. Training dengan logging progress
5. Export ke format GGUF (kuantisasi Q4_K_M) supaya ringan dipakai di Ollama
6. Instruksi download hasil untuk dipasang ke Ollama

> Jalankan di Colab dengan runtime **GPU** (Runtime > Change runtime type > T4 GPU atau lebih tinggi).

## 1. Install Unsloth dan dependency

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers triton
!pip install datasets

In [ ]:
import torch
print('CUDA tersedia:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Tidak ada GPU terdeteksi!')

## 2. Load base model Gemma 2B dengan Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None  # otomatis deteksi (Float16 untuk T4, Bfloat16 untuk A100+)
LOAD_IN_4BIT = True  # kuantisasi 4-bit agar hemat memori saat training

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print('Base model berhasil dimuat.')

## 3. Konfigurasi LoRA

`r=16`, `alpha=32`, dengan target modules standar untuk arsitektur Gemma (attention + MLP projections).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,  # 0 dioptimalkan oleh Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print('Adapter LoRA terpasang. Parameter yang dilatih:')
model.print_trainable_parameters()

## 4. Upload dan load `dataset-final.jsonl`

Upload file `dataset-final.jsonl` yang dihasilkan oleh `prepare-dataset.js` (gabungan `identity.jsonl` + materi kurikulum).

In [ ]:
from google.colab import files

print('Silakan upload file dataset-final.jsonl:')
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print('File diupload:', DATASET_PATH)

In [ ]:
from datasets import load_dataset

GEMMA_CHAT_TEMPLATE = """<start_of_turn>user
{instruction}<end_of_turn>
<start_of_turn>model
{output}<end_of_turn>"""

def format_prompt(example):
    example["text"] = GEMMA_CHAT_TEMPLATE.format(
        instruction=example["instruction"],
        output=example["output"],
    ) + tokenizer.eos_token
    return example

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.map(format_prompt)

print(f"Total sampel dataset: {len(dataset)}")
print("\nContoh sampel setelah diformat:\n")
print(dataset[0]["text"])

## 5. Training loop dengan progress logging

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

In [ ]:
print('=== Mulai training EduNusa ===')
training_stats = trainer.train()
print('\n=== Training selesai ===')
print(training_stats)

## 6. Simpan adapter LoRA (opsional, untuk backup)

In [ ]:
model.save_pretrained("edunusa-lora-adapter")
tokenizer.save_pretrained("edunusa-lora-adapter")
print('Adapter LoRA disimpan di folder edunusa-lora-adapter/')

## 7. Export ke format GGUF (kuantisasi Q4_K_M)

Q4_K_M dipilih karena memberi keseimbangan baik antara ukuran file (ringan) dan kualitas jawaban,
cocok untuk dijalankan di perangkat dengan sumber daya terbatas (sesuai konteks daerah 3T).

In [ ]:
model.save_pretrained_gguf(
    "edunusa-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print('Model berhasil diexport ke format GGUF (Q4_K_M) di folder edunusa-gguf/')

## 8. Download hasil GGUF

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("edunusa-gguf/*.gguf")
print('File GGUF yang ditemukan:', gguf_files)

for f in gguf_files:
    print(f'Mendownload {f} ...')
    files.download(f)

## 9. Cara memasang hasil fine-tuning ke Ollama

Setelah file `.gguf` (misalnya `edunusa-gguf/unsloth.Q4_K_M.gguf`) selesai didownload:

1. Ganti nama file menjadi `edunusa-finetuned.gguf` (opsional, agar lebih jelas), lalu salin ke folder `edunusa-model/` di project Belajar 3T.
2. Buka `edunusa-model/Modelfile`, ubah baris pertama dari:
   ```
   FROM gemma2:2b
   ```
   menjadi:
   ```
   FROM ./edunusa-finetuned.gguf
   ```
3. Build ulang model:
   ```bash
   cd edunusa-model
   ollama create edunusa -f Modelfile
   ```
4. Uji model:
   ```bash
   ollama run edunusa "Siapa kamu?"
   ```

Lihat `edunusa-model/README.md` bagian "Cara Ganti Base Weights dengan Hasil Fine-Tuning" untuk detail lebih lanjut.